# Diffusion: directly fit G, preserve D=GGᵀ
The five-PC teacher has general noise and one Brownian driver. The symbolic field is a (batch,5,1) factor, not a scalar magnitude or a diagonal noise model. Fit is performed only after the drift validation gate succeeds.

In [ ]:
from pathlib import Path
import os,sys,json
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
import pandas as pd,numpy as np
from IPython.display import display,Image,Markdown


In [ ]:
drift_result=json.loads(Path('outputs/sindy_pc5/drift/results.json').read_text())
assert drift_result['function_success'], 'Drift gate failed; diffusion is intentionally deferred'
root=Path('outputs/sindy_pc5/diffusion')
result=json.loads((root/'results.json').read_text())
display(pd.read_csv(root/'sweep.csv'))
display(pd.DataFrame([{ 'partition':name, **result[name]} for name in ('train','validation','test','test_in_domain')]))
print(result['selected']);print(result['threshold_jitter'])
display(Image(filename='outputs/figures/pc5_diffusion_sparsity.png'))

In [ ]:
from src.distill import PolynomialField,SymbolicSDE
import torch
f=PolynomialField.load(Path('outputs/sindy_pc5/drift/model.npz'))
g=PolynomialField.load(root/'model.npz')
model=SymbolicSDE(f,g)
x=torch.zeros((4,5),dtype=torch.float32)
G=model.g(torch.tensor(2.),x)
D=G@G.transpose(-1,-2)
print('G',tuple(G.shape),'D',tuple(D.shape),'minimum eigenvalue',float(torch.linalg.eigvalsh(D).min()))
display(Markdown((root/'equations.md').read_text()))

Each G component uses polynomial powers/interactions only. Forming D multiplies these fitted functions and can increase polynomial degree; this is covariance construction, not a new candidate library. PSD is structural, but it does not establish intrinsic biological noise or uniquely identified diffusion. Equality to a teacher factor is stronger than the identifiable covariance alone; another Brownian-factor gauge can represent the same law.